# 01 — Capa Bronze: ingesta de sismos y maestros
**Sismos** (carpeta `landing/sismos/<evento>/` del Volume): un archivo `.txt` de CISMID por estación. Es la fuente que llega cada vez que ocurre un sismo.
**Maestros** (`landing/maestros/`): Vs30, límites, microzonificación, Open Buildings y su altura 2.5D, muestra de STEAD, Norma E.030, censo, OSM, Tainan, Manta, BSSA14 y fragilidad GEM.

Reglas de Bronze: no se limpia nada; se agregan `_source_file`, `_evento_carga` (o `_fuente` en los maestros) e `_ingested_at`; la carga de sismos es **idempotente**: volver a ejecutar un sismo lo reemplaza, no lo duplica.

Los parámetros `fecha_desde` y `fecha_hasta` los entrega el job y eligen los sismos por su fecha. Los maestros se cargan la primera vez o con `recargar_maestros = true`.

## 1. Setup — imports

In [ ]:
%pip install -q geopandas rasterio h5py pypdf pyproj shapely seaborn

In [ ]:
import json
import re
import zipfile
from datetime import datetime
from pathlib import Path

import geopandas as gpd
import h5py
import numpy as np
import pandas as pd
import rasterio
from pypdf import PdfReader
from rasterio.windows import from_bounds

from pyspark.sql import functions as F

## 2. Catálogo, esquema y parámetros

In [ ]:
catalogo = "grupo06proyectofinal"
esquema = "sismos"
volumen = f"/Volumes/{catalogo}/{esquema}"

spark.sql(f"USE CATALOG {catalogo}")
spark.sql(f"USE SCHEMA {esquema}")
spark.conf.set("spark.sql.session.timeZone", "America/Lima")

dbutils.widgets.text("fecha_desde", "2007-01-01")
dbutils.widgets.text("fecha_hasta", "2025-12-31")
dbutils.widgets.text("recargar_maestros", "false")
fecha_desde, fecha_hasta = dbutils.widgets.get("fecha_desde"), dbutils.widgets.get("fecha_hasta")
recargar_maestros = dbutils.widgets.get("recargar_maestros").lower() == "true"

In [ ]:
PROVINCIAS_ESTUDIO = ["LIMA", "CALLAO"]
DISTRITOS_ESTUDIO = {"150104": "BARRANCO", "150122": "MIRAFLORES", "150130": "SAN BORJA", "150131": "SAN ISIDRO",
                     "150140": "SANTIAGO DE SURCO", "150141": "SURQUILLO"}
ZONA_E030_LIMA = 4
EPSG_UTM_LIMA = 32718
SEED = 700
FILTRO_HZ = (0.1, 25.0)
AMORTIGUAMIENTO = 0.05
PERIODOS = np.round(np.geomspace(0.05, 4.0, 50), 4)
G_CM_S2 = 980.665

## 3. Funciones de escritura

In [ ]:
def a_spark(pdf, esquema=None):
    pdf = pdf.copy()
    for c in pdf.columns:
        if isinstance(pdf[c].dtype, (pd.StringDtype, pd.ArrowDtype)):
            pdf[c] = pdf[c].astype(object)
        if pdf[c].dtype == object:
            pdf[c] = pdf[c].map(lambda v: v if v is None or isinstance(v, (str, bytes, list, dict, np.ndarray))
                                else (None if pd.isna(v) else str(v)))
    return spark.createDataFrame(pdf, esquema) if esquema is not None else spark.createDataFrame(pdf)


def escribir(df, tabla, particion=None, modo="overwrite", contar=True):
    escritor = df.write.format("delta").mode(modo)
    if modo == "overwrite":
        escritor = escritor.option("overwriteSchema", "true")
    if particion:
        escritor = escritor.partitionBy(particion)
    escritor.saveAsTable(tabla)
    if contar:
        n = spark.table(tabla).count()
        print(f"  {tabla}: {n:,} filas")
        return n


def subir_pandas(lotes, tabla, esquema=None, preparar=None, particion=None, mb_lote=64):
    if isinstance(lotes, pd.DataFrame):
        lotes = [lotes]
    modo, n_lotes = "overwrite", 0
    for pdf in lotes:
        if len(pdf) == 0:
            continue
        partes = max(1, int(np.ceil(pdf.memory_usage(deep=True).sum() / 1e6 / mb_lote)))
        for idx in np.array_split(np.arange(len(pdf)), partes):
            sdf = a_spark(pdf.iloc[idx], esquema)
            if preparar is not None:
                sdf = preparar(sdf)
            escribir(sdf, tabla, particion, modo, contar=False)
            modo, n_lotes = "append", n_lotes + 1
    n = spark.table(tabla).count()
    print(f"  {tabla}: {n:,} filas ({n_lotes} lotes)")
    return n


def inventario(prefijo):
    filas = []
    for t in spark.sql("SHOW TABLES").collect():
        if t.isTemporary or not t.tableName.startswith(prefijo):
            continue
        d = spark.sql(f"DESCRIBE DETAIL {t.tableName}").first()
        filas.append({"tabla": t.tableName, "filas": spark.table(t.tableName).count(), "mb": round((d["sizeInBytes"] or 0) / 1e6, 1)})
    return pd.DataFrame(filas)

In [ ]:
def con_metadatos(df, fuente, archivo=None):
    if archivo is not None or "_source_file" not in df.columns:
        df = df.withColumn("_source_file", F.lit(None if archivo is None else str(archivo)).cast("string"))
    return df.withColumn("_fuente", F.lit(fuente)).withColumn("_ingested_at", F.current_timestamp())


def nombres_validos(columnas):
    vistos, salida = {}, []
    for c in columnas:
        nombre = re.sub(r"[^0-9A-Za-z_]+", "_", str(c).strip()).strip("_") or "columna"
        vistos[nombre] = vistos.get(nombre, 0) + 1
        salida.append(nombre if vistos[nombre] == 1 else f"{nombre}_{vistos[nombre]}")
    return salida


def subir_crudo(pdf, tabla, fuente, archivo=None):
    pdf = pdf.copy()
    pdf.columns = nombres_validos(pdf.columns)
    pdf = pdf.astype(object).where(pdf.notna(), None)
    esquema = ", ".join(f"`{c}` string" for c in pdf.columns)
    return subir_pandas(pdf, f"bronze_{tabla}", esquema=esquema, preparar=lambda df: con_metadatos(df, fuente, archivo))

## 4. Sismos que llegaron al Volume
Cada carpeta de `landing/sismos/` es un sismo; su nombre empieza con la fecha. Se cargan los que caen entre `fecha_desde` y `fecha_hasta`.

In [ ]:
landing = f"{volumen}/landing"
MAESTROS = Path(f"{landing}/maestros")
dbutils.fs.mkdirs(f"{landing}/sismos")
eventos = sorted(e.name.rstrip("/") for e in dbutils.fs.ls(f"{landing}/sismos") if fecha_desde <= e.name[:10] <= fecha_hasta)
print("Sismos a procesar:", eventos)

In [ ]:
ESQUEMA_CISMID = "evento string, archivo_estacion string, bytes long, codificacion string, contenido string, _source_file string, _evento_carga string"

for ev in eventos:
    filas = []
    for txt in sorted(Path(f"{landing}/sismos/{ev}").glob("*.txt")):
        datos = txt.read_bytes()
        try:
            contenido, codificacion = datos.decode("utf-8"), "utf-8"
        except UnicodeDecodeError:
            contenido, codificacion = datos.decode("latin-1"), "latin-1"
        filas.append({"evento": ev, "archivo_estacion": txt.stem, "bytes": len(datos), "codificacion": codificacion,
                      "contenido": contenido, "_source_file": txt.name, "_evento_carga": ev})
    cismid = a_spark(pd.DataFrame(filas), ESQUEMA_CISMID).withColumn("_ingested_at", F.current_timestamp())
    if spark.catalog.tableExists("bronze_cismid_registros"):
        cismid.write.mode("overwrite").option("replaceWhere", f"_evento_carga = '{ev}'").saveAsTable("bronze_cismid_registros")
    else:
        cismid.write.saveAsTable("bronze_cismid_registros")
    print("bronze_cismid_registros", ev, len(filas))

## 5. Maestros
Datos de referencia que no cambian con cada sismo. Se cargan la primera vez o cuando `recargar_maestros = true`.

In [ ]:
cargar_maestros = recargar_maestros or not spark.catalog.tableExists("bronze_open_buildings")
print("Cargar maestros:", cargar_maestros)

### Vs30

In [ ]:
if cargar_maestros:
    limites = gpd.read_file(MAESTROS / "limites_inei" / "peru_distrital_simple.geojson")
    x0, y0, x1, y1 = limites[limites.NOMBPROV.isin(PROVINCIAS_ESTUDIO)].total_bounds
    MARGEN = 0.05
    archivo_vs30 = MAESTROS / "vs30" / "global_vs30.grd"

    with rasterio.open(archivo_vs30) as src:
        ventana = from_bounds(x0 - MARGEN, y0 - MARGEN, x1 + MARGEN, y1 + MARGEN, transform=src.transform).round_offsets().round_lengths()
        vs30 = src.read(1, window=ventana)
        tr = src.window_transform(ventana)
        res = src.res[0]
        fila0, col0 = int(ventana.row_off), int(ventana.col_off)
        nodata = src.nodata
        origen_lon, origen_lat = src.transform.c, src.transform.f

    filas, cols = np.indices(vs30.shape)
    pix = pd.DataFrame({
        "fila": (filas + fila0).ravel(), "col": (cols + col0).ravel(),
        "lon": (tr.c + (cols + 0.5) * tr.a).ravel(), "lat": (tr.f + (filas + 0.5) * tr.e).ravel(),
        "vs30": vs30.ravel().astype("float64"),
    })
    pix["grid_origen_lon"], pix["grid_origen_lat"], pix["grid_res"] = origen_lon, origen_lat, res
    print(f"Ventana {vs30.shape} | nodata={nodata} | Vs30 min/med/max = {np.nanmin(vs30):.0f}/{np.nanmedian(vs30):.0f}/{np.nanmax(vs30):.0f} m/s")
    subir_pandas(pix, "bronze_vs30_pixeles", preparar=lambda df: con_metadatos(df, "vs30_usgs", archivo_vs30.as_posix()))

### Microzonificación

In [ ]:
if cargar_maestros:
    paginas = []
    for pdf in sorted((MAESTROS / "microzonificacion").glob("*.pdf")):
        lector = PdfReader(pdf)
        distrito = pdf.stem.replace("microzonificacion_", "")
        for i, pag in enumerate(lector.pages, start=1):
            paginas.append({"distrito": distrito, "pagina": i, "n_paginas": len(lector.pages),
                            "texto": pag.extract_text() or "", "_source_file": pdf.as_posix()})
    pdf_df = pd.DataFrame(paginas)
    subir_pandas(pdf_df, "bronze_microzonificacion_paginas", preparar=lambda df: con_metadatos(df, "microzonificacion"))
    display(pdf_df.groupby("distrito").agg(paginas=("pagina", "max"), caracteres=("texto", lambda s: s.str.len().sum())))

### Open Buildings

In [ ]:
if cargar_maestros:
    archivos_ob = sorted({p for patron in ("**/*.csv", "**/*.csv.gz") for p in (MAESTROS / "open_buildings").glob(patron) if p.is_file()})
    print("Archivos:", [a.as_posix() for a in archivos_ob])
    TIPOS_OB = {"latitude": "float64", "longitude": "float64", "area_in_meters": "float64", "confidence": "float64",
                "geometry": str, "full_plus_code": str}
    ESQUEMA_OB = ("latitude double, longitude double, area_in_meters double, confidence double, geometry string, "
                  "full_plus_code string, _source_file string")


    def lotes_open_buildings(filas_lote=250_000):
        for archivo in archivos_ob:
            for pdf in pd.read_csv(archivo, usecols=list(TIPOS_OB), dtype=TIPOS_OB, chunksize=filas_lote):
                pdf = pdf[list(TIPOS_OB)].copy()
                pdf["_source_file"] = archivo.as_posix()
                yield pdf


    subir_pandas(lotes_open_buildings(), "bronze_open_buildings", esquema=ESQUEMA_OB,
                 preparar=lambda df: con_metadatos(df, "open_buildings_v3"))

### Open Buildings 2.5D

In [ ]:
if cargar_maestros:
    teselas = []
    for tif in sorted((MAESTROS / "open_buildings_25d").glob("*.tif")):
        with rasterio.open(tif) as src:
            h = src.read(1, out_shape=(src.height // 8, src.width // 8))
            b = src.bounds
            valido = h[(h != src.nodata) & (h > 0)]
            teselas.append({
                "tesela": tif.stem, "epsg": src.crs.to_epsg(), "res_m": src.res[0],
                "ancho": src.width, "alto": src.height,
                "xmin": b.left, "ymin": b.bottom, "xmax": b.right, "ymax": b.top, "nodata": float(src.nodata),
                "pct_px_con_edificio": round(100 * valido.size / h.size, 3),
                "altura_p50_m": float(np.median(valido)) if valido.size else None,
                "altura_p99_m": float(np.percentile(valido, 99)) if valido.size else None,
                "anio": int(re.search(r"_(\d{4})_\d\d_\d\d", tif.stem).group(1)),
                "_source_file": tif.as_posix(),
            })
    subir_pandas(pd.DataFrame(teselas), "bronze_open_buildings_25d_teselas", preparar=lambda df: con_metadatos(df, "open_buildings_25d"))

### Límites distritales

In [ ]:
if cargar_maestros:
    lim = limites.copy()
    lim["geometry_wkt"] = lim.geometry.to_wkt()
    lim = pd.DataFrame(lim.drop(columns="geometry"))
    lim["FECHA"] = lim["FECHA"].astype(str)
    subir_pandas(lim, "bronze_limites_inei",
                 preparar=lambda df: con_metadatos(df, "limites_inei", (MAESTROS / "limites_inei" / "peru_distrital_simple.geojson").as_posix()))
    archivo_comp = MAESTROS / "limites_inei" / "complemento_osm_la_punta_santa_anita.geojson"
    if archivo_comp.exists():
        comp = gpd.read_file(archivo_comp)
        comp["geometry_wkt"] = comp.geometry.to_wkt()
        subir_pandas(pd.DataFrame(comp.drop(columns="geometry")), "bronze_limites_complemento_osm",
                     preparar=lambda df: con_metadatos(df, "osm", archivo_comp.as_posix()))

In [ ]:
if cargar_maestros:
    archivo_osm_dist = MAESTROS / "limites_osm" / "distritos_lima_callao_osm.geojson"
    if archivo_osm_dist.exists():
        od = gpd.read_file(archivo_osm_dist)
        od["geometry_wkt"] = od.geometry.to_wkt()
        subir_pandas(pd.DataFrame(od.drop(columns="geometry")).astype({"ubigeo": str, "osm_id": "int64"}), "bronze_limites_osm_distritos",
                     preparar=lambda df: con_metadatos(df, "osm", archivo_osm_dist.as_posix()))
    else:
        print("Falta", archivo_osm_dist.as_posix(), "(ver README): silver usará los límites simplificados del INEI")

### STEAD (muestra de 10 000 trazas)

In [ ]:
if cargar_maestros:
    carpeta_stead = MAESTROS / "stead"
    stead = pd.read_csv(carpeta_stead / "chunk2_muestra.csv", low_memory=False)
    stead["_archivo_hdf5"] = (carpeta_stead / "chunk2_muestra.hdf5").as_posix()
    n_stead = subir_pandas(stead, "bronze_stead_metadata",
                           preparar=lambda df: con_metadatos(df, "stead", (carpeta_stead / "chunk2_muestra.csv").as_posix()))

    hdf5 = carpeta_stead / "chunk2_muestra.hdf5"
    if hdf5.exists():
        with h5py.File(hdf5, "r") as h:
            n_trazas = len(h["data"])
            ejemplo = next(iter(h["data"]))
            forma = h["data"][ejemplo].shape
        print(f"HDF5: {n_trazas:,} trazas (CSV: {n_stead:,}) | {'coinciden' if n_trazas == n_stead else 'NO coinciden'} | forma de una traza: {forma}")

### Norma E.030

In [ ]:
if cargar_maestros:
    for nombre in ["e030_zonas", "e030_factor_suelo", "e030_perfiles"]:
        archivo = MAESTROS / "norma_e030" / f"{nombre}.csv"
        subir_pandas(pd.read_csv(archivo), f"bronze_{nombre}", preparar=lambda df, a=archivo: con_metadatos(df, "norma_e030", a.as_posix()))

### Censo 2017

In [ ]:
if cargar_maestros:
    archivo_censo = MAESTROS / "censo2017" / "paredes_distrito_lima_callao.csv"
    if archivo_censo.exists():
        subir_pandas(pd.read_csv(archivo_censo, dtype={"ubigeo": str}), "bronze_censo2017_paredes",
                     preparar=lambda df: con_metadatos(df, "inei_censo2017_redatam", archivo_censo.as_posix()))
    else:
        print("Falta", archivo_censo.as_posix(), "(ver README)")

### OSM: número de pisos

In [ ]:
if cargar_maestros:
    archivo_osm = MAESTROS / "osm_niveles" / "lima_building_levels.json"
    if archivo_osm.exists():
        elementos = json.loads(archivo_osm.read_text(encoding="utf-8"))["elements"]
        osm = pd.DataFrame([{"osm_id": str(el["id"]), "lon": el["center"]["lon"], "lat": el["center"]["lat"],
                             "niveles_txt": el["tags"].get("building:levels")} for el in elementos if "center" in el])
        osm["niveles"] = pd.to_numeric(osm.niveles_txt, errors="coerce")
        subir_pandas(osm, "bronze_osm_niveles", preparar=lambda df: con_metadatos(df, "osm", archivo_osm.as_posix()))
    else:
        print("Falta", archivo_osm.as_posix(), "(ver README): los pisos salen solo de la altura 2.5D")

### Tainan

In [ ]:
if cargar_maestros:
    NS_XLSX = "{http://schemas.openxmlformats.org/spreadsheetml/2006/main}"


    def hoja_xlsx(ruta):
        from xml.etree import ElementTree as ET
        z = zipfile.ZipFile(ruta)
        compartidas = ["".join(t.text or "" for t in si.iter(NS_XLSX + "t")) for si in ET.fromstring(z.read("xl/sharedStrings.xml")).findall(NS_XLSX + "si")]
        filas = []
        for n, fila in enumerate(ET.fromstring(z.read("xl/worksheets/sheet1.xml")).iter(NS_XLSX + "row"), start=1):
            f = {"libro": ruta.name, "fila": str(n)}
            for c in fila.findall(NS_XLSX + "c"):
                v = c.find(NS_XLSX + "v")
                if v is not None:
                    f[re.match(r"[A-Z]+", c.get("r")).group(0)] = compartidas[int(v.text)] if c.get("t") == "s" else v.text
            filas.append(f)
        return pd.DataFrame(filas)


    carpeta_tainan = MAESTROS / "tainan_2016"
    for tabla, archivo, codificacion in (("tainan_edificios", "edificios/parameters_taiwan_eq_20170828.csv", "utf-8"),
                                         ("tainan_casos", "edificios/2016_Taiwan_Meinong_20170814.csv", "latin-1"),
                                         ("tainan_archivos", "edificios/1267_file_metadata.csv", "latin-1")):
        subir_crudo(pd.read_csv(carpeta_tainan / archivo, dtype=str, encoding=codificacion, encoding_errors="ignore"), tabla, "purr_tainan", (carpeta_tainan / archivo).as_posix())
    subir_crudo(pd.concat([hoja_xlsx(carpeta_tainan / "ondas" / n) for n in ("list_of_ground_motion.xlsx", "list_of_ground_motion_Tainan_City.xlsx")], ignore_index=True),
                "tainan_lista_registros", "purr_tainan")


    def lotes_tainan(n_lote=40):
        z = zipfile.ZipFile(carpeta_tainan / "ondas" / "Taiwan_CWB_ground_motion_data.zip")
        nombres = [n for n in z.namelist() if n.endswith(".txt") and not n.startswith("__MACOSX")]
        for i in range(0, len(nombres), n_lote):
            yield pd.DataFrame([{"archivo": n, "contenido": z.read(n).decode("latin-1")} for n in nombres[i:i + n_lote]])


    subir_pandas(lotes_tainan(), "bronze_tainan_registros", esquema="archivo string, contenido string", preparar=lambda df: con_metadatos(df, "purr_tainan"))

### Ecuador: Portoviejo y Manta

In [ ]:
if cargar_maestros:
    archivo_ecuador = MAESTROS / "portoviejo_2016" / "campo" / "parameters_priority_index_metadata.csv"
    subir_crudo(pd.read_csv(archivo_ecuador, dtype=str, encoding_errors="ignore"), "ecuador_edificios", "purr_ecuador", archivo_ecuador.as_posix())
    subir_pandas(pd.DataFrame([{"archivo": p.name, "contenido": p.read_bytes().decode("latin-1")} for p in sorted((MAESTROS / "manta_2016" / "ondas" / "datos").rglob("*.txt"))]), "bronze_manta_registros", esquema="archivo string, contenido string", preparar=lambda df: con_metadatos(df, "purr_ecuador"))

### Vs30 de Tainan y Manta

In [ ]:
if cargar_maestros:
    VENTANAS_CIUDAD = {"Tainan": (119.6, 22.2, 121.0, 23.8), "Manta": (-81.0, -1.2, -80.4, -0.7)}
    partes_vs30 = []
    with rasterio.open(archivo_vs30) as src:
        for ciudad, caja in VENTANAS_CIUDAD.items():
            ventana_c = from_bounds(*caja, transform=src.transform).round_offsets().round_lengths()
            v = src.read(1, window=ventana_c)
            tr_c = src.window_transform(ventana_c)
            filas_c, cols_c = np.indices(v.shape)
            partes_vs30.append(pd.DataFrame({"ciudad": ciudad, "fila": (filas_c + int(ventana_c.row_off)).ravel(), "col": (cols_c + int(ventana_c.col_off)).ravel(),
                                             "lon": (tr_c.c + (cols_c + 0.5) * tr_c.a).ravel(), "lat": (tr_c.f + (filas_c + 0.5) * tr_c.e).ravel(), "vs30": v.ravel().astype("float64")}))
    subir_pandas(pd.concat(partes_vs30, ignore_index=True), "bronze_vs30_pixeles_ciudades", preparar=lambda df: con_metadatos(df, "vs30_usgs", archivo_vs30.as_posix()))

### BSSA14 y fragilidad GEM

In [ ]:
if cargar_maestros:
    archivo_bssa = MAESTROS / "bssa14" / "bssa14_no_lineal.csv"
    subir_crudo(pd.read_csv(archivo_bssa, dtype=str), "bssa14_no_lineal", "bssa14", archivo_bssa.as_posix())
    subir_pandas(pd.DataFrame([{"archivo": p.name, "contenido": p.read_text(encoding="utf-8")} for p in sorted((MAESTROS / "fragilidad_gem").glob("*.csv"))]), "bronze_fragilidad_gem", esquema="archivo string, contenido string", preparar=lambda df: con_metadatos(df, "gem"))

## 6. Manifiesto del Volume

In [ ]:
manifiesto = pd.DataFrame([
    {"archivo": p.relative_to(landing).as_posix(), "carpeta": p.relative_to(landing).parts[1], "bytes": p.stat().st_size,
     "modificado": datetime.fromtimestamp(p.stat().st_mtime).isoformat(timespec="seconds")}
    for p in sorted(Path(landing).rglob("*")) if p.is_file()])
subir_pandas(manifiesto, "bronze_landing_manifiesto")
display(manifiesto.groupby("carpeta").agg(archivos=("archivo", "count"), gb=("bytes", lambda b: round(b.sum() / 1e9, 2))))

## 7. Verificación

In [ ]:
spark.table("bronze_cismid_registros").groupBy("_evento_carga", "codificacion").count().orderBy("_evento_carga").show(truncate=False)
inv = inventario("bronze_")
print(f"Tablas: {len(inv)} | filas: {inv.filas.sum():,} | tamaño: {inv.mb.sum():,.0f} MB")
display(inv)